# Prompt Engineering for Materials Science

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

Before building complex AI agents, it's worth understanding the simplest form of
"adapting" a foundation model to your domain: **prompt engineering**. By carefully
crafting how you ask questions, you can dramatically improve an LLM's performance on
materials science tasks — without any fine-tuning or infrastructure.

In this notebook, you'll see how different prompting strategies progressively improve
Claude's ability to:

1. Extract structured data from materials science text
2. Identify crystalline phases from peak descriptions
3. Reason about materials properties and characterization results

## Prompting Strategies (from simple to powerful)

| Strategy | Idea | Effort |
|----------|------|--------|
| **Zero-shot** | Just ask the question | None |
| **Few-shot** | Show examples first | Low |
| **Chain-of-thought** | Ask the model to reason step-by-step | Low |
| **Structured output** | Request a specific output format (JSON) | Low |
| **Domain context** | Inject domain knowledge via system prompt | Medium |

By the end, you'll know **when each strategy is sufficient** — and when you need
heavier tools (RAG, fine-tuning, agents).

## Setup

In [ ]:
# Install dependencies (uncomment if running in Colab)
# !pip install anthropic

In [ ]:
import os
import getpass
import json

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

API_KEY = os.environ["ANTHROPIC_API_KEY"]
print("API key configured.")

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=API_KEY)

def ask_claude(user_message, system_prompt=None, model="claude-sonnet-4-6", max_tokens=2048):
    """Simple helper to send a message to Claude and get the response text."""
    kwargs = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": user_message}],
    }
    if system_prompt:
        kwargs["system"] = system_prompt
    response = client.messages.create(**kwargs)
    return response.content[0].text

print("Client ready.")

---
## The Task: Extract Information from Materials Science Text

We'll use a consistent task throughout this notebook so you can see how each
prompting strategy improves the results.

**Task**: Given a paragraph from a materials science paper, extract structured
information about the material, synthesis conditions, and measured properties.

In [ ]:
# Sample paragraphs from (fictional) materials science papers

PARAGRAPHS = [
    {
        "text": (
            "Barium titanate (BaTiO3) thin films were deposited on SrTiO3 (001) "
            "substrates by pulsed laser deposition at a substrate temperature of "
            "750°C and an oxygen partial pressure of 100 mTorr. X-ray diffraction "
            "confirmed the epitaxial growth of the tetragonal phase with a c/a ratio "
            "of 1.011. The remnant polarization measured by P-E hysteresis was "
            "12.5 µC/cm² at an applied field of 200 kV/cm. The dielectric constant "
            "at 1 kHz was measured to be 1200."
        ),
        "expected": {
            "material": "BaTiO3",
            "substrate": "SrTiO3 (001)",
            "synthesis_method": "Pulsed laser deposition",
            "temperature_C": 750,
            "phase": "tetragonal",
            "properties": {
                "c_a_ratio": 1.011,
                "remnant_polarization_uC_cm2": 12.5,
                "dielectric_constant_1kHz": 1200,
            },
        },
    },
    {
        "text": (
            "Nanocrystalline TiO2 powders were synthesized via sol-gel processing "
            "using titanium isopropoxide as a precursor. The gel was calcined at "
            "450°C for 2 hours in air, yielding phase-pure anatase with an average "
            "crystallite size of 12 nm as determined by Scherrer analysis of the "
            "(101) reflection. BET surface area was 95 m²/g. UV-Vis spectroscopy "
            "showed a band gap of 3.2 eV, consistent with bulk anatase. "
            "Photocatalytic degradation of methylene blue under UV illumination "
            "showed 92% degradation after 60 minutes."
        ),
        "expected": {
            "material": "TiO2",
            "synthesis_method": "Sol-gel",
            "precursor": "Titanium isopropoxide",
            "temperature_C": 450,
            "phase": "anatase",
            "properties": {
                "crystallite_size_nm": 12,
                "surface_area_m2_g": 95,
                "band_gap_eV": 3.2,
                "photocatalytic_degradation_pct": 92,
            },
        },
    },
    {
        "text": (
            "High-entropy alloy CoCrFeMnNi was prepared by arc melting under argon "
            "atmosphere and subsequently cold-rolled to 80% reduction. Annealing at "
            "800°C for 1 hour produced a fully recrystallized FCC single-phase "
            "microstructure with an average grain size of 15 µm. Tensile testing at "
            "room temperature yielded an ultimate tensile strength of 650 MPa, "
            "yield strength of 350 MPa, and elongation to failure of 45%. "
            "Vickers hardness was measured at 165 HV."
        ),
        "expected": {
            "material": "CoCrFeMnNi (Cantor alloy)",
            "synthesis_method": "Arc melting + cold rolling + annealing",
            "temperature_C": 800,
            "phase": "FCC single-phase",
            "properties": {
                "grain_size_um": 15,
                "UTS_MPa": 650,
                "yield_strength_MPa": 350,
                "elongation_pct": 45,
                "hardness_HV": 165,
            },
        },
    },
]

print(f"Loaded {len(PARAGRAPHS)} test paragraphs.")
print(f"\nExample paragraph (first 200 chars):\n{PARAGRAPHS[0]['text'][:200]}...")

---
## Strategy 1: Zero-Shot Prompting

Just ask the question directly — no examples, no special instructions.

In [ ]:
# Zero-shot: just ask
paragraph = PARAGRAPHS[0]["text"]

response = ask_claude(
    f"Extract the material name, synthesis method, temperature, crystal phase, "
    f"and key measured properties from this paragraph:\n\n{paragraph}"
)
print(response)

**Observations:**
- The model usually gets the key facts right
- But the output format is unpredictable — sometimes bullet points, sometimes prose
- Property names and units may be inconsistent
- Hard to parse programmatically

Let's improve this with examples.

---
## Strategy 2: Few-Shot Prompting

Show Claude a couple of examples of the input-output format you want. This is one of the most effective and easiest improvements you can make.

In [ ]:
# Few-shot: provide examples first
few_shot_prompt = """Extract structured information from materials science paragraphs.

Here are two examples:

--- Example 1 ---
Paragraph: "ZnO nanowires were grown on sapphire substrates by chemical vapor deposition at 900°C. XRD confirmed the wurtzite phase. The band gap was 3.37 eV and the room-temperature photoluminescence showed a strong UV emission at 380 nm."

Extracted information:
- Material: ZnO
- Synthesis: Chemical vapor deposition
- Substrate: Sapphire
- Temperature: 900°C
- Phase: Wurtzite
- Properties: band gap = 3.37 eV, PL emission at 380 nm

--- Example 2 ---
Paragraph: "Polycrystalline SnO2 films were deposited by RF sputtering at 300°C onto glass substrates. The rutile phase was confirmed by XRD with a preferred (110) orientation. Electrical measurements showed a resistivity of 5×10⁻³ Ω·cm and a carrier concentration of 2×10²⁰ cm⁻³."

Extracted information:
- Material: SnO2
- Synthesis: RF sputtering
- Substrate: Glass
- Temperature: 300°C
- Phase: Rutile (preferred (110) orientation)
- Properties: resistivity = 5e-3 Ω·cm, carrier concentration = 2e20 cm⁻³

--- Your turn ---
Paragraph: "{paragraph}"

Extracted information:"""

paragraph = PARAGRAPHS[0]["text"]
response = ask_claude(few_shot_prompt.format(paragraph=paragraph))
print(response)

**Observations:**
- Output now follows a consistent format
- All key fields are present
- But it's still text — let's make it machine-readable.

---
## Strategy 3: Structured Output (JSON)

Ask Claude to return JSON. This makes the output programmatically parseable — critical for building data pipelines.

In [ ]:
structured_prompt = """Extract information from the following materials science paragraph.

Return ONLY a JSON object with this exact structure:
{{
    "material": "chemical formula",
    "synthesis_method": "method name",
    "substrate": "substrate or null",
    "temperature_C": number or null,
    "phase": "crystal phase",
    "characterization_techniques": ["list", "of", "techniques"],
    "properties": {{
        "property_name": {{"value": number, "unit": "unit string"}}
    }}
}}

Paragraph:
{paragraph}"""

paragraph = PARAGRAPHS[0]["text"]
response = ask_claude(structured_prompt.format(paragraph=paragraph))
print(response)

# Try to parse it
try:
    # Strip markdown code fences if present
    cleaned = response.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:-1])
    data = json.loads(cleaned)
    print("\n--- Successfully parsed as JSON ---")
    print(json.dumps(data, indent=2))
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

**Let's test this on all three paragraphs:**

In [ ]:
for i, para in enumerate(PARAGRAPHS):
    print(f"{'='*60}")
    print(f"Paragraph {i+1}")
    print(f"{'='*60}")

    response = ask_claude(structured_prompt.format(paragraph=para["text"]))

    cleaned = response.strip()
    if cleaned.startswith("```"):
        cleaned = "\n".join(cleaned.split("\n")[1:-1])

    try:
        data = json.loads(cleaned)
        print(f"Material: {data['material']}")
        print(f"Method:   {data['synthesis_method']}")
        print(f"Phase:    {data['phase']}")
        print(f"Properties: {len(data.get('properties', {}))} extracted")
        print(json.dumps(data, indent=2))
    except json.JSONDecodeError:
        print("Failed to parse JSON")
        print(response[:200])
    print()

---
## Strategy 4: Chain-of-Thought Prompting

Ask Claude to **reason step-by-step** before giving its answer. This improves accuracy
on tasks that require inference or domain reasoning — not just extraction.

Let's try a harder task: given XRD peak positions, identify the phase.
This requires the model to draw on its knowledge of crystallography.

In [ ]:
cot_prompt = """I measured an XRD pattern (Cu Kα radiation) from an unknown powder sample.
The major peak positions (2θ) are: 28.4°, 47.3°, 56.1°, 69.1°, 76.4°

What crystalline phase is this? Think through this step-by-step:
1. Consider the peak positions and their relative spacing
2. Think about what crystal systems produce these reflections
3. Consider common materials that match this pattern
4. Give your final identification with confidence level"""

response = ask_claude(cot_prompt)
print(response)

Now compare with a **zero-shot** version of the same question:

In [ ]:
# Same question, no chain-of-thought
simple_prompt = (
    "I measured XRD peaks (Cu Kα) at 2θ = 28.4°, 47.3°, 56.1°, 69.1°, 76.4°. "
    "What material is this?"
)

response = ask_claude(simple_prompt)
print(response)

**Observations:**
- Chain-of-thought produces more detailed, reasoned answers
- The model explains *why* it reaches its conclusion, not just *what*
- Particularly valuable for ambiguous cases or when you need to verify the reasoning
- In an agentic workflow, CoT helps the model make better tool-calling decisions

---
## Strategy 5: Domain Context Injection

The **system prompt** lets you inject persistent domain knowledge that shapes all of
Claude's responses. Think of it as giving the model a "briefing" before it starts working.

This is the most powerful prompting strategy for domain adaptation — it's essentially
giving the model a crash course in your specific field.

In [ ]:
# A domain-aware system prompt for XRD analysis
DOMAIN_SYSTEM_PROMPT = """You are an expert X-ray diffraction analyst at a materials \
characterization facility. You specialize in phase identification using Cu Kα radiation \
(λ = 1.5406 Å).

When analyzing XRD data, always:
1. Report peaks using standard 2θ notation
2. Reference JCPDS/ICDD card numbers when possible
3. Consider both major and minor phases
4. Note any peak splitting that might indicate phase transitions
5. Flag potential issues like preferred orientation, peak broadening, or overlapping reflections

Common reference peaks you should know (Cu Kα):
- Si: 28.44° (111), 47.30° (220), 56.12° (311)
- Cu: 43.30° (111), 50.43° (200), 74.13° (220)
- Al: 38.47° (111), 44.74° (200), 65.09° (220)
- TiO2 rutile: 27.45° (110), 36.09° (101), 54.32° (211)
- TiO2 anatase: 25.28° (101), 37.80° (004), 48.05° (200)
- BaTiO3: 22.17° (100), 31.51° (110), 45.17° (200)

When you identify a phase, include:
- Confidence level (high/medium/low)
- Miller indices for matched reflections
- Any unmatched peaks that might indicate impurities"""

# Now ask about a tricky case: a mixture
mixture_prompt = (
    "I have an XRD pattern from a thin film sample with peaks at: "
    "25.3°, 28.4°, 37.8°, 47.3°, 48.1°, 56.1°. "
    "The film was deposited on a silicon substrate. "
    "Identify all phases present."
)

# Without domain context
print("=" * 60)
print("WITHOUT domain context:")
print("=" * 60)
response_plain = ask_claude(mixture_prompt)
print(response_plain)

print("\n")

# With domain context
print("=" * 60)
print("WITH domain context:")
print("=" * 60)
response_domain = ask_claude(mixture_prompt, system_prompt=DOMAIN_SYSTEM_PROMPT)
print(response_domain)

**Key differences with domain context:**
- More precise identification using standard reference data
- Reports Miller indices for each matched peak
- Distinguishes substrate peaks from film peaks
- Provides confidence levels
- Uses domain-standard terminology and conventions

This is the simplest form of domain adaptation — and it's often sufficient for
many practical tasks. When it's not enough, you move to:
- **RAG** (retrieve relevant papers/data on the fly)
- **Fine-tuning** (train the model on your domain data)
- **Agents** (give the model tools to look things up)

---
## Comparison: Which Strategy When?

| Strategy | Best for | Limitations |
|----------|---------|-------------|
| **Zero-shot** | Quick, one-off questions | Inconsistent format, may miss details |
| **Few-shot** | Standardizing output format | Need good examples, limited by context window |
| **Chain-of-thought** | Complex reasoning, ambiguous data | Slower, more verbose, more tokens |
| **Structured output** | Data pipelines, automated processing | Model may break JSON format |
| **Domain context** | Persistent domain expertise | Limited by context window size |

### Decision Framework

```
Is the task simple and well-defined?
├── Yes → Zero-shot or Few-shot
└── No → Does it require reasoning?
    ├── Yes → Chain-of-thought + Domain context
    └── No, but it's domain-specific → Domain context

Do you need machine-readable output?
├── Yes → Structured output (JSON)
└── No → Free-form text is fine

Is the knowledge static or does it change?
├── Static → System prompt
└── Dynamic → You need RAG (retrieval)

Does the task require multi-step actions?
├── Yes → You need an Agent (see `04_agent_microscopy.ipynb`)
└── No → Prompting strategies are sufficient
```

---
## Exercises

1. **Try a different domain**: Modify the system prompt for a technique you use
   (Raman, XRF, TEM, etc.) and test it on a relevant question.

2. **Stress test structured output**: Find an edge case where the JSON extraction
   fails. How would you make it more robust?

3. **Combine strategies**: Use few-shot examples + chain-of-thought + structured output
   together. Does combining them help or hurt?

In [ ]:
# Exercise space — try your own prompts here!

# Example: modify this for your own characterization technique
my_system_prompt = """You are an expert in [YOUR TECHNIQUE] analysis.
When analyzing data, always:
1. ...
2. ...
"""

# my_question = "..."
# response = ask_claude(my_question, system_prompt=my_system_prompt)
# print(response)

---
## Next Steps

You've seen how prompting alone can go a long way in adapting foundation models to
materials science. But when you need the model to **take actions** — load data,
run analyses, query databases — you need an **agent**.

**Head to `04_agent_microscopy.ipynb`** to learn how to build a scientific agent with tool use.